# Week 4: ViT Training Pipeline for CIFAR-10

## Problem Setup

**Goal**: Build a clean, reproducible Vision Transformer training pipeline for CIFAR-10 that optimizes **wall-clock time to target accuracy** (e.g., 94% test accuracy).

**Success metrics**:
- Time-to-target accuracy (primary)
- Throughput (images/sec)
- Peak GPU memory
- Accuracy curves over time

**Data**: CIFAR-10 (50k train, 10k test, 32×32 RGB, 10 classes)

**Constraints**: Reproducible runs, clear logging, trustworthy measurements for later "mod" experiments.

### Mathematical Formulation

**Objective**: Minimize cross-entropy loss over training data:
$$\mathcal{L} = -\frac{1}{N} \sum_{i=1}^{N} \sum_{c=1}^{C} y_{i,c} \log \hat{p}_{i,c}$$

where $y_{i,c}$ is the one-hot label and $\hat{p}_{i,c}$ is the softmax output for class $c$.

**ViT forward pass**:
1. Patch embedding: $x \in \mathbb{R}^{B \times 3 \times 32 \times 32} \rightarrow z \in \mathbb{R}^{B \times 65 \times d}$ (64 patches + [CLS])
2. Transformer: $z \leftarrow \text{TransformerBlock}(z)$ for $L$ layers
3. Classification: $\hat{y} = \text{Linear}(z[:, 0])$ (use [CLS] token)

## Implementation

### Imports and Setup

In [1]:
import sys
from pathlib import Path

# Project root: works whether notebook is run from STAT4830/ or STAT4830/notebooks/
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
DATA_DIR = PROJECT_ROOT / "data"

import torch
from src.model import VisionTransformer, count_parameters
from src.model import count_parameters
from src.utils import (
    get_device,
    get_cifar10_loaders,
    get_model,
    get_pretrained_vit,
    set_seed,
    train,
    validate,
    get_peak_gpu_memory_mb,
    reset_peak_gpu_memory,
)

print(f"PyTorch: {torch.__version__}")
device = get_device()
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch: 2.10.0
Device: mps


### Configuration and Reproducibility

**USE_PRETRAINED=True** loads ImageNet ViT, fine-tunes until 94% accuracy. Primary metric: **time-to-target** (wall-clock sec to reach 94%).

In [2]:
# Reproducibility
SEED = 42
set_seed(SEED)

# Use pre-trained ViT (timm) - trains until 94% accuracy or max_epochs
USE_PRETRAINED = True
TARGET_ACCURACY = 94.0
MAX_EPOCHS = 100  # Cap in case target not reached

# Config
BATCH_SIZE = 64 if USE_PRETRAINED else 128
LR = 3e-4
PATCH_SIZE = 4
EMBED_DIM = 96
DEPTH = 4
NUM_HEADS = 4

# For pre-trained: img_size=224, ImageNet norm. For scratch: img_size=32, CIFAR norm.
IMG_SIZE = 224 if USE_PRETRAINED else 32

device = get_device()
print(f"USE_PRETRAINED={USE_PRETRAINED} | TARGET={TARGET_ACCURACY}% | img_size={IMG_SIZE}")

USE_PRETRAINED=True | TARGET=94.0% | img_size=224


### Data Loading

CIFAR-10 with RandomCrop+RandomHorizontalFlip. For pre-trained ViT: resize to 224×224 and ImageNet normalization.

In [3]:
print(f"Data directory: {DATA_DIR.resolve()}")

train_loader, test_loader = get_cifar10_loaders(
    data_dir=str(DATA_DIR),
    batch_size=BATCH_SIZE,
    num_workers=4,
    augment_train=True,
    img_size=IMG_SIZE,
)

print(f"Train batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")
sample_batch, sample_labels = next(iter(train_loader))
print(f"Batch shape: {sample_batch.shape}, Labels shape: {sample_labels.shape}")

Data directory: /Users/ayushtripathi/STAT4830/data


/Users/ayushtripathi/STAT4830/.venv/lib/python3.14/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Train batches: 782
Test batches: 157
Batch shape: torch.Size([64, 3, 224, 224]), Labels shape: torch.Size([64])


### Model Creation

Pre-trained ViT (timm) or scratch ViT. Pre-trained uses ImageNet weights; head replaced for 10 classes.

In [4]:
if USE_PRETRAINED:
    model = get_pretrained_vit(
        model_name="vit_tiny_patch16_224",
        num_classes=10,
        pretrained=True,
    )
else:
    model = get_model(
        patch_size=PATCH_SIZE,
        embed_dim=EMBED_DIM,
        depth=DEPTH,
        num_heads=NUM_HEADS,
        dropout=0.0,
    )

n_params = count_parameters(model)
print(f"Model parameters: {n_params:,}")
print(f"Pre-trained: {USE_PRETRAINED}")

# Sanity check: forward pass
reset_peak_gpu_memory()
model = model.to(device)
_ = model(sample_batch.to(device))
print(f"Output shape: {_.shape}")  # (B, 10)

/Users/ayushtripathi/STAT4830/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model parameters: 5,526,346
Pre-trained: True
Output shape: torch.Size([64, 10])


### Training Loop

Trains until **94% test accuracy** (primary metric: time-to-target) or max epochs. Early stops when target reached.

In [ ]:
reset_peak_gpu_memory()

results = train(
    model,
    train_loader,
    test_loader,
    num_epochs=MAX_EPOCHS,
    lr=LR,
    device=device,
    log_every=1,
    target_accuracy=TARGET_ACCURACY,
)

ttt = results.get("time_to_target")
print(f"\n--- Summary ---")
print(f"Best test accuracy: {results['best_acc']:.2f}%")
print(f"Time-to-target ({TARGET_ACCURACY}%): {ttt:.2f}s" if ttt is not None else f"Time-to-target ({TARGET_ACCURACY}%): Not reached")
print(f"Total wall-clock time: {results['total_time']:.2f}s")
print(f"Peak GPU memory: {results.get('peak_gpu_mb', 'N/A')} MB")

Python(28332) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


## Validation

### Accuracy Curves and Resource Metrics

Plot train/test accuracy and throughput over time.

In [ ]:
import matplotlib.pyplot as plt

history = results["history"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Accuracy
axes[0].plot(history["train_acc"], label="Train")
axes[0].plot(history["test_acc"], label="Test")
axes[0].axhline(y=94, color="gray", linestyle="--", label="Target 94%")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy (%)")
axes[0].set_title("Accuracy over time")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Throughput
axes[1].plot(history["throughput"], color="green")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Images/sec")
axes[1].set_title("Training throughput")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Test Cases

1. **Output shape**: Model outputs (B, 10) logits for CIFAR-10.
2. **Reproducibility**: Same seed yields same initial weights and training trajectory.
3. **Resource monitoring**: Peak GPU memory and throughput logged per epoch.

In [ ]:
# Test: output shape
model.eval()
with torch.no_grad():
    out = model(sample_batch.to(device))
assert out.shape == (sample_batch.size(0), 10), f"Expected (B, 10), got {out.shape}"

# Test: loss is finite
criterion = torch.nn.CrossEntropyLoss()
loss = criterion(out, sample_labels.to(device))
assert torch.isfinite(loss), "Loss must be finite"

print("✓ Output shape correct")
print("✓ Loss finite")

## Documentation and Next Steps

**Known limitations**:
- Baseline uses simple augmentations (RandomCrop, RandomHorizontalFlip). RandAugment/Mixup could improve accuracy.
- 10 epochs may not reach 94%; more epochs or tuning may be needed.
- Deterministic mode may slow training; consider benchmarking with `cudnn.benchmark=True` for production runs.

**Next steps**:
- Add early stopping at target accuracy (94%) to measure time-to-target.
- Implement recipe configs (YAML/JSON) for systematic mod experiments.
- Add mixed precision (AMP) as first speed mod.